In [3]:
# Find-S & Candidate Elimination | Version Space Impossibility | Employee Promotion Dataset

import pandas as pd

def covers(h, x):
    for i in range(len(x)):
        if h[i] != '?' and h[i] != '∅' and h[i] != x[i]:
            return False
    return True

def is_more_general(h1, h2):
    for i in range(len(h1)):
        if h1[i] != '?' and h1[i] != h2[i] and h2[i] != '∅':
            return False
    return True

In [4]:
print("="*70)
print("DATASET: Employee Promotion Eligibility")
print("="*70)

print("\nDataset Examples:")
print("-"*70)
print(f"{'Sit':<4} {'Situation':<25} {'Experience':<12} {'Performance':<12} {'Skills':<12} {'Label':<10}")
print("-"*70)

dataset_examples = [
    (1, "Identical diff labels", "5+ years", "Excellent", "Advanced", "Positive"),
    (1, "Identical diff labels", "5+ years", "Excellent", "Advanced", "Negative"),
    (2, "Positive ⊏ Negative", "Engineering", "A", "-", "Positive"),
    (2, "Positive ⊏ Negative", "Engineering", "?", "-", "Negative"),
    (3, "XOR pattern", "Yes", "High", "-", "Positive"),
    (3, "XOR pattern", "No", "Low", "-", "Positive"),
    (3, "XOR pattern", "Yes", "Low", "-", "Negative"),
    (3, "XOR pattern", "No", "High", "-", "Negative"),
    (4, "Missing attribute", "Masters", "High", "-", "Positive"),
    (4, "Missing attribute", "Masters", "Low", "-", "Negative"),
    (4, "Missing attribute", "Bachelor", "High", "-", "Negative"),
    (5, "Sequential contradiction", "Completed", "Good", "High", "Positive"),
    (5, "Sequential contradiction", "Completed", "Good", "Low", "Negative"),
    (5, "Sequential contradiction", "Pending", "Good", "High", "Positive"),
    (5, "Sequential contradiction", "Completed", "Good", "High", "Negative"),
]

for sit, name, a1, a2, a3, label in dataset_examples:
    print(f"{sit:<4} {name:<25} {a1:<12} {a2:<12} {a3:<12} {label:<10}")

print("\n" + "="*70)
print("Attribute Description:")
print("-"*70)
print("| Attribute     | Values                                      |")
print("|---------------|---------------------------------------------|")
print("| Experience    | 5+ years, <5 years                          |")
print("| Performance   | Excellent, Good, Average                    |")
print("| Skills        | Advanced, Intermediate, Beginner            |")
print("| Department    | Engineering, Sales, HR                      |")
print("| Rating        | A, B, C, D                                  |")
print("| Certification | Yes, No                                     |")
print("| Education     | Masters, Bachelor                           |")
print("| Salary        | High, Low                                   |")
print("| Project       | Completed, Pending                          |")
print("| Teamwork      | Good, Poor                                  |")
print("| Initiative    | High, Low                                   |")
print("| Label         | Positive (Eligible) / Negative (Not Eligible)|")
print("="*70)

DATASET: Employee Promotion Eligibility

Dataset Examples:
----------------------------------------------------------------------
Sit  Situation                 Experience   Performance  Skills       Label     
----------------------------------------------------------------------
1    Identical diff labels     5+ years     Excellent    Advanced     Positive  
1    Identical diff labels     5+ years     Excellent    Advanced     Negative  
2    Positive ⊏ Negative       Engineering  A            -            Positive  
2    Positive ⊏ Negative       Engineering  ?            -            Negative  
3    XOR pattern               Yes          High         -            Positive  
3    XOR pattern               No           Low          -            Positive  
3    XOR pattern               Yes          Low          -            Negative  
3    XOR pattern               No           High         -            Negative  
4    Missing attribute         Masters      High         -            

In [5]:
def find_s(examples, attrs):
    S = ['∅'] * len(attrs)
    for vals, label in examples:
        if label == 'Positive':
            for i in range(len(attrs)):
                if S[i] == '∅':
                    S[i] = vals[i]
                elif S[i] != vals[i]:
                    S[i] = '?'
    return S

In [6]:
def candidate_elimination(examples, attrs):
    S = ['∅'] * len(attrs)
    G = [['?'] * len(attrs)]
    trace = []

    for idx, (vals, label) in enumerate(examples, 1):
        if label == 'Positive':
            # Update S
            for i in range(len(attrs)):
                if S[i] == '∅':
                    S[i] = vals[i]
                elif S[i] != vals[i]:
                    S[i] = '?'

            # Remove from G any hypothesis that doesn't cover positive
            G = [h for h in G if covers(h, vals)]

        else:  # Negative
            # Specialize G to exclude negative
            new_G = []
            for h in G:
                if covers(h, vals):
                    # Replace each '?' with value from negative example
                    for i in range(len(attrs)):
                        if h[i] == '?':
                            new_h = h.copy()
                            new_h[i] = vals[i]
                            new_G.append(new_h)
                else:
                    new_G.append(h)

            # Remove duplicates
            unique = []
            for h in new_G:
                if h not in unique:
                    unique.append(h)
            new_G = unique

            # Remove hypotheses that are more specific than another
            G_temp = []
            for h in new_G:
                is_specific = False
                for other in new_G:
                    if h != other and all(other[i] == '?' or other[i] == h[i] for i in range(len(attrs))):
                        is_specific = True
                        break
                if not is_specific:
                    G_temp.append(h)

            # Remove any hypothesis that still covers the negative
            G = [h for h in G_temp if not covers(h, vals)]

            # Remove any hypothesis not more general than S
            G = [h for h in G if all(S[i] == '∅' or S[i] == '?' or h[i] == '?' or h[i] == S[i] for i in range(len(attrs)))]

        trace.append((idx, vals, label, S.copy(), [h.copy() for h in G]))

        # Check if version space empty
        if not G:
            return S, G, trace, False

        # Check if S covers any negative seen so far
        for ex_vals, ex_label in examples[:idx]:
            if ex_label == 'Negative' and covers(S, ex_vals):
                return S, G, trace, False

    return S, G, trace, True

In [7]:
print("="*70)
print("SITUATION 1: Identical Examples with Different Labels")
print("="*70)

attrs = ['Experience', 'Performance', 'Skills']
examples = [
    (['5+ years', 'Excellent', 'Advanced'], 'Positive'),
    (['5+ years', 'Excellent', 'Advanced'], 'Negative')
]

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

# Check if S covers negative
for v, l in examples:
    if l == 'Negative' and covers(S, v):
        print(f"  WARNING: S covers negative → INCONSISTENT")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")
        print(f"    >>> VERSION SPACE EMPTY <<<")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] Same exact example as Positive and Negative → contradiction")

SITUATION 1: Identical Examples with Different Labels

[INPUT]
  E1: {'Experience': '5+ years', 'Performance': 'Excellent', 'Skills': 'Advanced'} -> Positive
  E2: {'Experience': '5+ years', 'Performance': 'Excellent', 'Skills': 'Advanced'} -> Negative

[FIND-S]
  Final S: {'Experience': '5+ years', 'Performance': 'Excellent', 'Skills': 'Advanced'}

[CANDIDATE ELIMINATION]

  After E1: {'Experience': '5+ years', 'Performance': 'Excellent', 'Skills': 'Advanced'} -> Positive
    S = {'Experience': '5+ years', 'Performance': 'Excellent', 'Skills': 'Advanced'}
    G = [{'Experience': '?', 'Performance': '?', 'Skills': '?'}]

  After E2: {'Experience': '5+ years', 'Performance': 'Excellent', 'Skills': 'Advanced'} -> Negative
    S = {'Experience': '5+ years', 'Performance': 'Excellent', 'Skills': 'Advanced'}
    G = []
    >>> VERSION SPACE EMPTY <<<

[RESULT] Version Space: EMPTY

[WHY] Same exact example as Positive and Negative → contradiction


In [8]:
print("="*70)
print("SITUATION 2: Positive More Specific than Negative")
print("="*70)

attrs = ['Department', 'Rating']
examples = [
    (['Engineering', 'A'], 'Positive'),
    (['Engineering', '?'], 'Negative')
]

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")
        print(f"    >>> VERSION SPACE EMPTY <<<")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] Positive (Engineering,A) is specialization of negative (Engineering,?)")
print("      Any hypothesis covering positive also covers negative.")

SITUATION 2: Positive More Specific than Negative

[INPUT]
  E1: {'Department': 'Engineering', 'Rating': 'A'} -> Positive
  E2: {'Department': 'Engineering', 'Rating': '?'} -> Negative

[FIND-S]
  Final S: {'Department': 'Engineering', 'Rating': 'A'}

[CANDIDATE ELIMINATION]

  After E1: {'Department': 'Engineering', 'Rating': 'A'} -> Positive
    S = {'Department': 'Engineering', 'Rating': 'A'}
    G = [{'Department': '?', 'Rating': '?'}]

  After E2: {'Department': 'Engineering', 'Rating': '?'} -> Negative
    S = {'Department': 'Engineering', 'Rating': 'A'}
    G = []
    >>> VERSION SPACE EMPTY <<<

[RESULT] Version Space: EMPTY

[WHY] Positive (Engineering,A) is specialization of negative (Engineering,?)
      Any hypothesis covering positive also covers negative.


In [9]:
print("="*70)
print("SITUATION 3: XOR Pattern")
print("="*70)

attrs = ['Certification', 'Experience']
examples = [
    (['Yes', 'High'], 'Positive'),
    (['No', 'Low'], 'Positive'),
    (['Yes', 'Low'], 'Negative'),
    (['No', 'High'], 'Negative')
]

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] XOR needs disjunction (OR) but hypothesis space only conjunctions (AND)")

SITUATION 3: XOR Pattern

[INPUT]
  E1: {'Certification': 'Yes', 'Experience': 'High'} -> Positive
  E2: {'Certification': 'No', 'Experience': 'Low'} -> Positive
  E3: {'Certification': 'Yes', 'Experience': 'Low'} -> Negative
  E4: {'Certification': 'No', 'Experience': 'High'} -> Negative

[FIND-S]
  Final S: {'Certification': '?', 'Experience': '?'}

[CANDIDATE ELIMINATION]

  After E1: {'Certification': 'Yes', 'Experience': 'High'} -> Positive
    S = {'Certification': 'Yes', 'Experience': 'High'}
    G = [{'Certification': '?', 'Experience': '?'}]

  After E2: {'Certification': 'No', 'Experience': 'Low'} -> Positive
    S = {'Certification': '?', 'Experience': '?'}
    G = [{'Certification': '?', 'Experience': '?'}]

  After E3: {'Certification': 'Yes', 'Experience': 'Low'} -> Negative
    S = {'Certification': '?', 'Experience': '?'}
    G = []

[RESULT] Version Space: EMPTY

[WHY] XOR needs disjunction (OR) but hypothesis space only conjunctions (AND)


In [10]:
print("="*70)
print("SITUATION 4: Missing Crucial Attribute")
print("="*70)

attrs = ['Education', 'Salary']
examples = [
    (['Masters', 'High'], 'Positive'),
    (['Masters', 'Low'], 'Negative'),
    (['Bachelor', 'High'], 'Negative')
]

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")
        print(f"    >>> VERSION SPACE EMPTY <<<")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] True concept depends on missing attribute (YearsInCompany)")
print("      No hypothesis using only Education and Salary works.")

SITUATION 4: Missing Crucial Attribute

[INPUT]
  E1: {'Education': 'Masters', 'Salary': 'High'} -> Positive
  E2: {'Education': 'Masters', 'Salary': 'Low'} -> Negative
  E3: {'Education': 'Bachelor', 'Salary': 'High'} -> Negative

[FIND-S]
  Final S: {'Education': 'Masters', 'Salary': 'High'}

[CANDIDATE ELIMINATION]

  After E1: {'Education': 'Masters', 'Salary': 'High'} -> Positive
    S = {'Education': 'Masters', 'Salary': 'High'}
    G = [{'Education': '?', 'Salary': '?'}]

  After E2: {'Education': 'Masters', 'Salary': 'Low'} -> Negative
    S = {'Education': 'Masters', 'Salary': 'High'}
    G = []
    >>> VERSION SPACE EMPTY <<<

[RESULT] Version Space: EMPTY

[WHY] True concept depends on missing attribute (YearsInCompany)
      No hypothesis using only Education and Salary works.


In [11]:
print("="*70)
print("SITUATION 5: Sequential Contradiction")
print("="*70)

attrs = ['Project', 'Teamwork', 'Initiative']
examples = [
    (['Completed', 'Good', 'High'], 'Positive'),
    (['Completed', 'Good', 'Low'], 'Negative'),
    (['Pending', 'Good', 'High'], 'Positive'),
    (['Completed', 'Good', 'High'], 'Negative')
]

print("\n[INPUT]")
for i, (v, l) in enumerate(examples, 1):
    print(f"  E{i}: {dict(zip(attrs, v))} -> {l}")

print("\n[FIND-S]")
S = find_s(examples, attrs)
print(f"  Final S: {dict(zip(attrs, S))}")

print("\n[CANDIDATE ELIMINATION]")
S_f, G_f, trace, possible = candidate_elimination(examples, attrs)

for t in trace:
    idx, vals, lbl, S_state, G_state = t
    print(f"\n  After E{idx}: {dict(zip(attrs, vals))} -> {lbl}")
    print(f"    S = {dict(zip(attrs, S_state))}")
    if G_state:
        print(f"    G = {[dict(zip(attrs, h)) for h in G_state]}")
    else:
        print(f"    G = []")
    if not G_state:
        print(f"    >>> EMPTY <<<")

print(f"\n[RESULT] Version Space: {'EMPTY' if not possible else 'EXISTS'}")
print("\n[WHY] E4 contradicts E1 after algorithm updated boundaries → contradiction mid-execution")

SITUATION 5: Sequential Contradiction

[INPUT]
  E1: {'Project': 'Completed', 'Teamwork': 'Good', 'Initiative': 'High'} -> Positive
  E2: {'Project': 'Completed', 'Teamwork': 'Good', 'Initiative': 'Low'} -> Negative
  E3: {'Project': 'Pending', 'Teamwork': 'Good', 'Initiative': 'High'} -> Positive
  E4: {'Project': 'Completed', 'Teamwork': 'Good', 'Initiative': 'High'} -> Negative

[FIND-S]
  Final S: {'Project': '?', 'Teamwork': 'Good', 'Initiative': 'High'}

[CANDIDATE ELIMINATION]

  After E1: {'Project': 'Completed', 'Teamwork': 'Good', 'Initiative': 'High'} -> Positive
    S = {'Project': 'Completed', 'Teamwork': 'Good', 'Initiative': 'High'}
    G = [{'Project': '?', 'Teamwork': '?', 'Initiative': '?'}]

  After E2: {'Project': 'Completed', 'Teamwork': 'Good', 'Initiative': 'Low'} -> Negative
    S = {'Project': 'Completed', 'Teamwork': 'Good', 'Initiative': 'High'}
    G = []
    >>> EMPTY <<<

[RESULT] Version Space: EMPTY

[WHY] E4 contradicts E1 after algorithm updated bounda

In [12]:
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

print("\nSituation 1: Identical diff labels → EMPTY")
print("Situation 2: Positive ⊏ Negative → EMPTY")
print("Situation 3: XOR pattern → EMPTY")
print("Situation 4: Missing attribute → EMPTY")
print("Situation 5: Sequential contradiction → EMPTY")

print("\n" + "="*70)
print("CONCLUSION: All 5 situations result in EMPTY VERSION SPACE")
print("="*70)


SUMMARY

Situation 1: Identical diff labels → EMPTY
Situation 2: Positive ⊏ Negative → EMPTY
Situation 3: XOR pattern → EMPTY
Situation 4: Missing attribute → EMPTY
Situation 5: Sequential contradiction → EMPTY

CONCLUSION: All 5 situations result in EMPTY VERSION SPACE
